# AI6103 Colab entry point

Running all cells with the defaults performs preparation only. Full training and the final test each require explicit opt-in. Upload (or clone) the whole repository to Google Drive, not just this Notebook. Put the dataset in `data/` or `data.zip` at the repository root. Checkpoints stay on Drive; dataset images may be copied to local Colab disk for speed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import sys, subprocess, shutil, zipfile

# Edit this location to the repository folder on YOUR Drive.
PROJECT = Path('/content/drive/MyDrive/AI6103-github')
assert (PROJECT / 'scripts/run_all.py').exists(), 'Upload the repository to Drive first'
# Results/checkpoints are durable, separate from Colab session storage.
OUTPUT = PROJECT / 'results'
FIGURES = PROJECT / 'figures'
REPORT = PROJECT / 'report'
# Use a local dataset copy for performance; do not modify the supplied splits.
DATA = Path('/content/ai6103_data/data')
if not DATA.exists():
    if (PROJECT / 'data').exists():
        shutil.copytree(PROJECT / 'data', DATA)
    else:
        archive = PROJECT / 'data.zip'
        assert archive.exists(), 'Place data.zip or data/ in PROJECT'
        destination = DATA.parent.resolve()
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            for member in z.infolist():
                assert (destination / member.filename).resolve().is_relative_to(destination), 'Unsafe archive path'
            z.extractall(destination)
assert (DATA / 'train_labels.csv').exists()

In [ ]:
# Keep Colab's matched torch/torchvision pair; install the other project tools.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT / 'requirements.txt')], check=True)
# Restart the runtime after installation if existing imported binary packages changed.
# Before official runs, record the working environment and do not change it mid-sweep.
def cli(*arguments, check=True):
    return subprocess.run([sys.executable, str(PROJECT / 'scripts/run_all.py'),
        '--data-root', str(DATA), '--output-root', str(OUTPUT),
        '--figure-root', str(FIGURES), *arguments], check=check)

In [ ]:
cli('prepare-stats')
cli('validate')
preflight = cli('preflight', check=False)
print('Preflight exit code:', preflight.returncode, '(2 means GPU unavailable or insufficient disk)')
cli('plot', '--beta-only')

## Explicit future training
Set `RUN_SECTION` to one of 3, 4, 5, 6 only when GPU resources are ready. Run sections in that order; dependencies and completion are verified. `--resume` restores partial experiments and skips only fully verified compatible completions. Set `EXPERIMENT` to an exact ID for one condition. If the runtime or package versions change, restore a matching environment before resuming. A stale lock left by a hard disconnect must be removed only after confirming the original process has ended.

In [ ]:
RUN_SECTION = None  # safe default: no training
EXPERIMENT = None
if RUN_SECTION is not None:
    assert preflight.returncode == 0, 'GPU preflight must pass'
    arguments = ['train', '--section', str(RUN_SECTION), '--resume']
    if EXPERIMENT is not None:
        arguments += ['--experiment', EXPERIMENT]
    cli(*arguments)

In [ ]:
SELECT_SECTION = None  # 3 for LR; 6 for the downstream setup
if SELECT_SECTION is not None:
    cli('select', '--section', str(SELECT_SECTION))

In [ ]:
GENERATE_EXPERIMENT_FIGURES = False
FREEZE_COMPLETE_EXPERIMENTS = False
if GENERATE_EXPERIMENT_FIGURES:
    cli('plot')
if FREEZE_COMPLETE_EXPERIMENTS:
    cli('freeze')

## Final hold-out evaluation: separate future operation
Requires all eight completed experiments and an immutable frozen manifest. The durable started marker prevents automatic repeated evaluation after a disconnect. Leave this disabled until all experiments are complete.

In [ ]:
AUTHORIZE_FINAL_TEST = False
if AUTHORIZE_FINAL_TEST:
    cli('evaluate-final', '--authorize-final-test')

In [ ]:
# Generate an honest draft; final generation additionally needs real results,
# final test, checked figures, an author and authored evidence-based discussion.
subprocess.run([sys.executable, str(PROJECT / 'scripts/fill_latex.py'),
    '--mode', 'draft', '--data-root', str(DATA), '--output-root', str(OUTPUT),
    '--figure-root', str(FIGURES), '--report-root', str(REPORT)], check=True)

In [ ]:
EXPORT_RESULTS = False
EXPORT_LARGE_CHECKPOINTS = False
if EXPORT_RESULTS:
    arguments = ['export'] + (['--checkpoints'] if EXPORT_LARGE_CHECKPOINTS else [])
    cli(*arguments)
    print('Small report_results.zip and optional large checkpoints.zip are under', OUTPUT / 'exports')
# No dataset archive and no automatic downloads are triggered.